In [32]:
# set project root and load the finalized SROIE modeling table
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sroie_model_df = pd.read_csv(PROJECT_ROOT / "outputs" / "sroie_model_df.csv")

print(sroie_model_df.shape)
display(
    sroie_model_df[
        [
            "doc_id",
            "strict_high_risk",
            "review_worthy",
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
        ]
    ].head()
)

(712, 48)


,doc_id,strict_high_risk,review_worthy,company_hard,date_hard,address_hard,total_hard,low_ocr_support
0,X00016469612,0,1,True,False,False,False,False
1,X00016469619,0,0,False,False,False,False,False
2,X00016469620,0,1,True,False,False,False,False
3,X00016469622,0,0,False,False,False,False,False
4,X00016469623,0,0,False,False,False,False,False


In [33]:
# inspect the two SROIE targets before evaluating rule baselines
display(
    pd.DataFrame({
        "strict_high_risk_rate": [sroie_model_df["strict_high_risk"].mean()],
        "review_worthy_rate": [sroie_model_df["review_worthy"].mean()],
    })
)

display(pd.crosstab(sroie_model_df["strict_high_risk"], sroie_model_df["review_worthy"]))

,strict_high_risk_rate,review_worthy_rate
0,0.066011,0.13764


review_worthy,0,1
strict_high_risk,,
0,614,51
1,0,47


In [34]:
# define reusable evaluation function for binary rule baselines
def evaluate_binary_baseline(y_true: pd.Series, y_pred: pd.Series, baseline_name: str) -> dict:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    tp = int((y_true & y_pred).sum())
    tn = int((~y_true & ~y_pred).sum())
    fp = int((~y_true & y_pred).sum())
    fn = int((y_true & ~y_pred).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    accuracy = (tp + tn) / len(y_true) if len(y_true) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "baseline": baseline_name,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
        "pred_verify_rate": float(y_pred.mean()),
    }

In [35]:
# build the baseline rule predictions once so we can score them against both targets
baseline_pred_df = pd.DataFrame({
    "doc_id": sroie_model_df["doc_id"],
    "always_accept": False,
    "always_verify": True,
    "verify_if_any_missing": sroie_model_df["any_field_missing"].astype(bool),
    "verify_if_ocr_empty": sroie_model_df["ocr_is_empty"].astype(bool),
    "verify_if_total_not_in_ocr": (~sroie_model_df["total_in_ocr"]).astype(bool),
    "verify_if_address_not_in_ocr": (~sroie_model_df["address_in_ocr"]).astype(bool),
    "verify_if_company_not_in_ocr": (~sroie_model_df["company_in_ocr"]).astype(bool),
    "verify_if_date_not_in_ocr": (~sroie_model_df["date_in_ocr"]).astype(bool),
    "verify_if_total_ambiguous": (sroie_model_df["exact_total_matches"] != 1).astype(bool),
    "verify_if_low_ocr_support": sroie_model_df["low_ocr_support"].astype(bool),
    "verify_if_amount_heavy": (sroie_model_df["n_amount_like_tokens"] >= 15).astype(bool),
    "verify_if_two_or_more_hard_flags": (
        sroie_model_df[["company_hard", "date_hard", "address_hard", "total_hard"]]
        .sum(axis=1) >= 2
    ),
    "verify_if_any_hard_flag": (
        sroie_model_df[["company_hard", "date_hard", "address_hard", "total_hard"]]
        .sum(axis=1) >= 1
    ),
    "verify_if_address_or_company_hard": (
        sroie_model_df["address_hard"] | sroie_model_df["company_hard"]
    ),
})
display(baseline_pred_df.head())

,doc_id,always_accept,always_verify,verify_if_any_missing,verify_if_ocr_empty,verify_if_total_not_in_ocr,verify_if_address_not_in_ocr,verify_if_company_not_in_ocr,verify_if_date_not_in_ocr,verify_if_total_ambiguous,verify_if_low_ocr_support,verify_if_amount_heavy,verify_if_two_or_more_hard_flags,verify_if_any_hard_flag,verify_if_address_or_company_hard
0,X00016469612,False,True,False,False,False,False,True,False,True,False,False,False,True,True
1,X00016469619,False,True,False,False,False,False,False,False,False,False,False,False,False,False
2,X00016469620,False,True,False,False,False,False,True,False,True,False,False,False,True,True
3,X00016469622,False,True,False,False,False,False,False,False,False,False,True,False,False,False
4,X00016469623,False,True,False,False,False,False,False,False,True,False,True,False,False,False


In [36]:
# score all baselines against the strict high-risk target
strict_rows = []

for col in baseline_pred_df.columns:
    if col == "doc_id":
        continue
    strict_rows.append(
        evaluate_binary_baseline(
            y_true=sroie_model_df["strict_high_risk"],
            y_pred=baseline_pred_df[col],
            baseline_name=col,
        )
    )

strict_results_df = pd.DataFrame(strict_rows).sort_values(
    ["f1", "recall", "precision", "accuracy"],
    ascending=False,
).reset_index(drop=True)

display(strict_results_df)

,baseline,tp,tn,fp,fn,precision,recall,specificity,accuracy,f1,pred_verify_rate
0,verify_if_two_or_more_hard_flags,47,665,0,0,1.000000,1.000000,1.000000,1.000000,1.000000,0.066011
1,verify_if_total_not_in_ocr,45,664,1,2,0.978261,0.957447,0.998496,0.995787,0.967742,0.064607
2,verify_if_date_not_in_ocr,45,662,3,2,0.937500,0.957447,0.995489,0.992978,0.947368,0.067416
3,verify_if_any_missing,43,663,2,4,0.955556,0.914894,0.996992,0.991573,0.934783,0.063202
4,verify_if_company_not_in_ocr,47,653,12,0,0.796610,1.000000,0.981955,0.983146,0.886792,0.082865
5,verify_if_address_not_in_ocr,47,627,38,0,0.552941,1.000000,0.942857,0.946629,0.712121,0.119382
6,verify_if_address_or_company_hard,47,615,50,0,0.484536,1.000000,0.924812,0.929775,0.652778,0.136236
7,verify_if_any_hard_flag,47,611,54,0,0.465347,1.000000,0.918797,0.924157,0.635135,0.141854
8,verify_if_ocr_empty,8,665,0,39,1.000000,0.170213,1.000000,0.945225,0.290909,0.011236
9,verify_if_low_ocr_support,11,589,76,36,0.126437,0.234043,0.885714,0.842697,0.164179,0.122191


In [37]:
# score all baselines against the broader review-worthy target
review_rows = []

for col in baseline_pred_df.columns:
    if col == "doc_id":
        continue
    review_rows.append(
        evaluate_binary_baseline(
            y_true=sroie_model_df["review_worthy"],
            y_pred=baseline_pred_df[col],
            baseline_name=col,
        )
    )

review_results_df = pd.DataFrame(review_rows).sort_values(
    ["f1", "recall", "precision", "accuracy"],
    ascending=False,
).reset_index(drop=True)

display(review_results_df)

,baseline,tp,tn,fp,fn,precision,recall,specificity,accuracy,f1,pred_verify_rate
0,verify_if_address_or_company_hard,97,614,0,1,1.000000,0.989796,1.000000,0.998596,0.994872,0.136236
1,verify_if_any_hard_flag,98,611,3,0,0.970297,1.000000,0.995114,0.995787,0.984925,0.141854
2,verify_if_address_not_in_ocr,85,614,0,13,1.000000,0.867347,1.000000,0.981742,0.928962,0.119382
3,verify_if_company_not_in_ocr,59,614,0,39,1.000000,0.602041,1.000000,0.945225,0.751592,0.082865
4,verify_if_two_or_more_hard_flags,47,614,0,51,1.000000,0.479592,1.000000,0.928371,0.648276,0.066011
5,verify_if_date_not_in_ocr,46,612,2,52,0.958333,0.469388,0.996743,0.924157,0.630137,0.067416
6,verify_if_total_not_in_ocr,45,613,1,53,0.978261,0.459184,0.998371,0.924157,0.625000,0.064607
7,verify_if_any_missing,44,613,1,54,0.977778,0.448980,0.998371,0.922753,0.615385,0.063202
8,verify_if_total_ambiguous,86,188,426,12,0.167969,0.877551,0.306189,0.384831,0.281967,0.719101
9,verify_if_low_ocr_support,23,550,64,75,0.264368,0.234694,0.895765,0.804775,0.248649,0.122191


In [38]:
# compare the same baselines side by side across the two SROIE targets
baseline_compare_df = (
    strict_results_df[["baseline", "precision", "recall", "f1", "pred_verify_rate"]]
    .rename(columns={
        "precision": "strict_precision",
        "recall": "strict_recall",
        "f1": "strict_f1",
        "pred_verify_rate": "strict_pred_rate",
    })
    .merge(
        review_results_df[["baseline", "precision", "recall", "f1", "pred_verify_rate"]]
        .rename(columns={
            "precision": "review_precision",
            "recall": "review_recall",
            "f1": "review_f1",
            "pred_verify_rate": "review_pred_rate",
        }),
        on="baseline",
        how="inner",
    )
    .sort_values(["review_f1", "strict_f1"], ascending=False)
    .reset_index(drop=True)
)

display(baseline_compare_df)

,baseline,strict_precision,strict_recall,strict_f1,strict_pred_rate,review_precision,review_recall,review_f1,review_pred_rate
0,verify_if_address_or_company_hard,0.484536,1.000000,0.652778,0.136236,1.000000,0.989796,0.994872,0.136236
1,verify_if_any_hard_flag,0.465347,1.000000,0.635135,0.141854,0.970297,1.000000,0.984925,0.141854
2,verify_if_address_not_in_ocr,0.552941,1.000000,0.712121,0.119382,1.000000,0.867347,0.928962,0.119382
3,verify_if_company_not_in_ocr,0.796610,1.000000,0.886792,0.082865,1.000000,0.602041,0.751592,0.082865
4,verify_if_two_or_more_hard_flags,1.000000,1.000000,1.000000,0.066011,1.000000,0.479592,0.648276,0.066011
5,verify_if_date_not_in_ocr,0.937500,0.957447,0.947368,0.067416,0.958333,0.469388,0.630137,0.067416
6,verify_if_total_not_in_ocr,0.978261,0.957447,0.967742,0.064607,0.978261,0.459184,0.625000,0.064607
7,verify_if_any_missing,0.955556,0.914894,0.934783,0.063202,0.977778,0.448980,0.615385,0.063202
8,verify_if_total_ambiguous,0.087891,0.957447,0.161002,0.719101,0.167969,0.877551,0.281967,0.719101
9,verify_if_low_ocr_support,0.126437,0.234043,0.164179,0.122191,0.264368,0.234694,0.248649,0.122191


In [39]:
# define the best simple rules for each SROIE target
best_strict_rule = "verify_if_two_or_more_hard_flags"
best_review_rule = "verify_if_address_or_company_hard"

print("best strict baseline:", best_strict_rule)
print("best review baseline:", best_review_rule)

display(
    baseline_compare_df.loc[
        baseline_compare_df["baseline"].isin([best_strict_rule, best_review_rule])
    ]
)

best strict baseline: verify_if_two_or_more_hard_flags
best review baseline: verify_if_address_or_company_hard


,baseline,strict_precision,strict_recall,strict_f1,strict_pred_rate,review_precision,review_recall,review_f1,review_pred_rate
0,verify_if_address_or_company_hard,0.484536,1.0,0.652778,0.136236,1.0,0.989796,0.994872,0.136236
4,verify_if_two_or_more_hard_flags,1.000000,1.0,1.000000,0.066011,1.0,0.479592,0.648276,0.066011


In [40]:
# try a few richer hand-built SROIE review rules to see whether anything beats address_or_company_hard
baseline_pred_df["review_rule_v2_address_or_company_or_total"] = (
    sroie_model_df["address_hard"]
    | sroie_model_df["company_hard"]
    | sroie_model_df["total_hard"]
)

baseline_pred_df["review_rule_v3_address_or_company_or_date_total_pair"] = (
    sroie_model_df["address_hard"]
    | sroie_model_df["company_hard"]
    | (
        sroie_model_df[["date_hard", "total_hard"]].sum(axis=1) >= 2
    )
)

baseline_pred_df["review_rule_v4_address_or_company_or_any_two_hard"] = (
    sroie_model_df["address_hard"]
    | sroie_model_df["company_hard"]
    | (
        sroie_model_df[["company_hard", "date_hard", "address_hard", "total_hard"]].sum(axis=1) >= 2
    )
)

baseline_pred_df["review_rule_v5_address_or_company_or_total_ambiguous"] = (
    sroie_model_df["address_hard"]
    | sroie_model_df["company_hard"]
    | (sroie_model_df["exact_total_matches"] != 1)
)

display(
    baseline_pred_df[
        [
            "review_rule_v2_address_or_company_or_total",
            "review_rule_v3_address_or_company_or_date_total_pair",
            "review_rule_v4_address_or_company_or_any_two_hard",
            "review_rule_v5_address_or_company_or_total_ambiguous",
        ]
    ]
    .mean()
    .to_frame("pred_rate")
)

,pred_rate
review_rule_v2_address_or_company_or_total,0.137640
review_rule_v3_address_or_company_or_date_total_pair,0.136236
review_rule_v4_address_or_company_or_any_two_hard,0.136236
review_rule_v5_address_or_company_or_total_ambiguous,0.734551


In [41]:
# evaluate the richer hand-built review rules against the broader review-worthy target
extra_review_rows = []

for col in [
    "review_rule_v2_address_or_company_or_total",
    "review_rule_v3_address_or_company_or_date_total_pair",
    "review_rule_v4_address_or_company_or_any_two_hard",
    "review_rule_v5_address_or_company_or_total_ambiguous",
]:
    extra_review_rows.append(
        evaluate_binary_baseline(
            y_true=sroie_model_df["review_worthy"],
            y_pred=baseline_pred_df[col],
            baseline_name=col,
        )
    )

extra_review_results_df = pd.DataFrame(extra_review_rows).sort_values(
    ["f1", "recall", "precision", "accuracy"],
    ascending=False,
).reset_index(drop=True)

display(extra_review_results_df)

,baseline,tp,tn,fp,fn,precision,recall,specificity,accuracy,f1,pred_verify_rate
0,review_rule_v3_address_or_company_or_date_tota...,97,614,0,1,1.000000,0.989796,1.000000,0.998596,0.994872,0.136236
1,review_rule_v4_address_or_company_or_any_two_hard,97,614,0,1,1.000000,0.989796,1.000000,0.998596,0.994872,0.136236
2,review_rule_v2_address_or_company_or_total,97,613,1,1,0.989796,0.989796,0.998371,0.997191,0.989796,0.137640
3,review_rule_v5_address_or_company_or_total_amb...,97,188,426,1,0.185468,0.989796,0.306189,0.400281,0.312399,0.734551


In [42]:
# compare the stronger review rules directly against the current best review baseline
review_rule_compare_df = pd.concat(
    [
        review_results_df.loc[
            review_results_df["baseline"].isin(
                [
                    "verify_if_address_or_company_hard",
                    "verify_if_address_not_in_ocr",
                    "verify_if_any_hard_flag",
                ]
            )
        ],
        extra_review_results_df,
    ],
    ignore_index=True,
).sort_values(
    ["f1", "recall", "precision", "accuracy"],
    ascending=False,
).reset_index(drop=True)

display(review_rule_compare_df)

,baseline,tp,tn,fp,fn,precision,recall,specificity,accuracy,f1,pred_verify_rate
0,verify_if_address_or_company_hard,97,614,0,1,1.000000,0.989796,1.000000,0.998596,0.994872,0.136236
1,review_rule_v3_address_or_company_or_date_tota...,97,614,0,1,1.000000,0.989796,1.000000,0.998596,0.994872,0.136236
2,review_rule_v4_address_or_company_or_any_two_hard,97,614,0,1,1.000000,0.989796,1.000000,0.998596,0.994872,0.136236
3,review_rule_v2_address_or_company_or_total,97,613,1,1,0.989796,0.989796,0.998371,0.997191,0.989796,0.137640
4,verify_if_any_hard_flag,98,611,3,0,0.970297,1.000000,0.995114,0.995787,0.984925,0.141854
5,verify_if_address_not_in_ocr,85,614,0,13,1.000000,0.867347,1.000000,0.981742,0.928962,0.119382
6,review_rule_v5_address_or_company_or_total_amb...,97,188,426,1,0.185468,0.989796,0.306189,0.400281,0.312399,0.734551


In [43]:
# inspect the 2 review-worthy receipts missed by address_or_company_hard
missed_by_best_review_rule_df = sroie_model_df.loc[
    (sroie_model_df["review_worthy"] == 1)
    & (~baseline_pred_df["verify_if_address_or_company_hard"]),
    [
        "doc_id",
        "strict_high_risk",
        "review_worthy",
        "company_hard",
        "date_hard",
        "address_hard",
        "total_hard",
        "low_ocr_support",
        "n_tokens",
        "ocr_word_count",
        "n_amount_like_tokens",
        "exact_total_matches",
        "company_in_ocr",
        "date_in_ocr",
        "address_in_ocr",
        "total_in_ocr",
    ],
].copy()

print(missed_by_best_review_rule_df.shape)
display(missed_by_best_review_rule_df)

(1, 16)


,doc_id,strict_high_risk,review_worthy,company_hard,date_hard,address_hard,total_hard,low_ocr_support,n_tokens,ocr_word_count,n_amount_like_tokens,exact_total_matches,company_in_ocr,date_in_ocr,address_in_ocr,total_in_ocr
687,X51008142038,0,1,False,True,False,False,True,29,74,4,1,True,False,True,True
